# מעבדה 18 — פלוקטואציות והובלה

במעבדה הזו הרעש, שבמשך שבעה-עשר מודולים הקורס מיצע אותו עד שנעלם, נרתם לעבודה. אתם קוראים קיבול חום
מתוך פלוקטואציות האנרגיה של מערכת במגע עם אמבט חום, בלי לחמם דבר; צופים בפלוקטואציה היחסית יורדת כמו
$N^{-1/2}$; משחררים עשרת אלפים הולכים אקראיים וצופים בהם נמסים אל הפתרון של משוואת הדיפוזיה; מודדים
את מקדם הדיפוזיה שלהם בשלוש דרכים בלתי תלויות; נותנים להם זיכרון של מהירותם וצופים בנוסחה הפשוטה
ביותר עבור $D$ נכשלת, בעוד האינטגרל של זיכרון המהירות אינו נכשל; צופים בגז של מודול 04 שאין בו
דיפוזיה כלל; מעריכים את המהלך החופשי הממוצע ואת מקדמי ההובלה של גזים ממשיים ובודקים אותם מול מדידות;
ולבסוף שוקלים אטום, כפי שעשה פרן, מתוך התנועה הבראונית של גרגר שרף.

המסלול: האובייקטים (חלק 1); קיבול חום מתוך הרעש (חלק 2); ריצה ארוכה יותר (חלק 3); חוק ה-$N^{-1/2}$
(חלק 4); ענן ההולכים (חלק 5); $D$ בשלוש דרכים (חלק 6); הולכים עם זיכרון (חלק 7); הקופסה של מודול 04,
שאין בה דיפוזיה (חלק 8); התנגשויות בגז ממשי (חלק 9); צמיגות ומוליכות חום מול המדידה (חלק 10);
האטומים של פרן (חלק 11); ניסוי דיפוזיה משלכם (חלק 12); הבדיקות האוטומטיות (חלק 13); וארבעה כלי חקירה
(חלק 14).

עברו עליה בסדר. במקום שבו המחברת מבקשת מכם לנבא, רשמו את ניבויכם בתא המיועד לכך **לפני** הרצת התא
הבא. ניבוי שהתחייבתם אליו הוא הדרך האמינה היחידה לגלות שטעיתם.

**יחידות.** יחידות SI לכל אורך הדרך. ההולכים חיים על ישר ונמדדים ביחידות משלהם: אורכים בצעדים $\ell$
וזמנים בזמני צעד $\tau$, כך ש-$D = \ell^2/2\tau$ שווה ל-$0.5$. הגזים של חלקים 9 ו-10 תלת-ממדיים,
ונמדדים במטרים ובשניות.

## מפרט המודל

| | |
|---|---|
| **מערכת** | הולכים אקראיים בלתי תלויים על ישר, המייצגים מולקולות מסומנות; ועבור הפלוקטואציות, הגז של מודול 04 והמערכות הדו-רמתיות של מודול 11 במגע עם אמבט חום |
| **דינמיקה** | צעדים של $\pm\ell$ בכל $\tau$, בלתי תלויים זה בזה, או צעדים החוזרים על הצעד האחרון בהסתברות $q$ ובכל מקרה אחר מוגרלים מחדש; האומדים משתמשים רק במיקומים ובמהירויות שנרשמו |
| **גבול** | ישר בלתי חסום עבור ההולכים; הקופסה הקשיחה של מודול 04 בחלק 8; אמבט חום ב-$T$ עבור הפלוקטואציות |
| **צבר** | הולכים בלתי תלויים מממשים את הצבר ישירות; האנרגיות הן הגרלות קנוניות, כל אחת בלתי תלויה בקודמתה |
| **מוזנח** | אינטראקציות בין ההולכים, הסעה וסחיפה, כל זיכרון מעבר להתמדה של צעד אחד |
| **תקף כאשר** | זמנים ארוכים בהשוואה לזיכרון המהירות; הולכים רבים; צעדים בעלי שונות סופית |
| **אופני כישלון** | צעדים מתואמים ($D = \ell^2/2\tau$ נכשל); זמנים מסדר הגודל של הזיכרון (תנועה בליסטית); סחיפה או הסעה; דפנות (הפיזור מגיע לרוויה) |

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from thermolab import gases, kinetics, partition, sampling, transport
from thermolab.constants import AMU, K_B, N_A
from thermolab.validation import scaling_exponent

try:
    import piplite  # noqa: F401
except ImportError:
    # Desktop / nbmake: the repository root is three directories up from this notebook.
    DATA = Path("..", "..", "..", "data")
else:
    # JupyterLite bundles only the notebooks/ tree (jupyter_lite_config.json's
    # LiteBuildConfig.contents), so the browser gets its own copy of the CSVs co-located here.
    DATA = Path("data")

M_AR = 39.948 * AMU   # an argon atom [kg]
GAP = 1e-21           # the two-level systems' gap [J]
STEP, TAU = 1.0, 1.0  # a walker's step and step time: walkers are measured in their own units


def read_table(name):
    # A data file as a dict of columns; every line starting with # is commentary.
    lines = [line for line in (DATA / name).read_text(encoding="utf-8").splitlines()
             if line and not line.startswith("#")]
    header = lines[0].split(",")
    columns = zip(*(line.split(",") for line in lines[1:]), strict=True)
    return {h: np.array(c) if h == "gas" else np.array(c, dtype=float)
            for h, c in zip(header, columns, strict=True)}


print(f"two-level gap / k_B = {GAP / K_B:.1f} K; walker D = l^2/(2 tau) = "
      f"{transport.persistent_walk_d(STEP, TAU)} l^2/tau")

## חלק 1 — האובייקטים

| פונקציה | מה היא נותנת |
|---|---|
| `gas_energy_samples(N, T, m, n, rng, ...)`, `two_level_energy_samples(N, gap, T, n, rng)` | `n` אנרגיות קנוניות בלתי תלויות של הגז של מודול 04 או של $N$ מערכות דו-רמתיות |
| `energy_fluctuation_cv(energies, T)` | $C_V = \sigma_E^2/\kB T^2$ עם פס השגיאה שלו |
| `walker_trajectories(n, steps, l, rng, persistence)`, `walker_velocities` | המיקומים של ההולכים בכל צעד, והמהירויות שלהם |
| `msd`, `diffusion_greens_function`, `step_profile` | ממוצע ריבוע ההעתק; הפתרונות של משוואת הדיפוזיה עבור מקור נקודתי ועבור שכבת צבע |
| `d_from_msd`, `d_from_histogram`, `d_from_vacf` | שלוש הערכות של $D$, כל אחת מהן `Estimate` עם `.value` ו-`.error` |
| `velocity_autocorrelation`, `green_kubo_running`, `persistent_walk_d` | $C(k)$; האינטגרל המצטבר שלו; ה-$D$ המדויק של הילוך בעל התמדה $q$ |
| `box_gas_trace` | הגז של מודול 04, נרשם צעד אחר צעד |
| `kinetics.mean_free_path`, `kinetics.collision_rate`, `kinetic_estimates` | $\lambda$, $\nu$ והשלישייה $D$, $\eta$, $\kappa$ של גז מדולל של כדורים קשיחים |
| `stokes_friction`, `einstein_diffusion`, `avogadro_from_diffusion` | גרר סטוקס, יחס איינשטיין, וה-$N_A$ של פרן |

לפני שתריצו משהו, רשמו את תשובותיכם לארבע השאלות שבעמוד המודול.

**הניבויים שלכם:**

1.
2.
3.
4.

## חלק 2 — קיבול חום מתוך הרעש

הגרילו את הגז של מודול 04 פעמים רבות, בכל פעם עם מהירויות חדשות מהתפלגות מקסוול-בולצמן ובלי לשנות את
קנה המידה שלהן (`fix_temperature=False`), כך שלאנרגיה שלו יש פלוקטואציות מעותק לעותק, כמו לאנרגיה של
מערכת קנונית. קראו את $C_V = \sigma_E^2/\kB T^2$ מתוך הפיזור. אחר כך עשו אותו דבר עבור $1000$ מערכות
דו-רמתיות בארבע טמפרטורות, והשוו לביטוי הסגור של מודול 12 ולקיבול חום שנמדד בדרך הרגילה: שינוי
הטמפרטורה ב-$\pm 5\%$ ומעקב אחר האנרגיה הממוצעת.

**הניבוי שלכם:** מה יהיה $C_V$ לפי הרעש עבור $100$ אטומי ארגון, ביחידות של $N\kB$? ומה עבור אותו גז
כשהוא מוגרל באופן מיקרו-קנוני, כשהאנרגיה שלו קבועה?

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
rng = np.random.default_rng(1801)
T = 300.0
argon = transport.energy_fluctuation_cv(transport.gas_energy_samples(100, T, M_AR, 10_000, rng), T)
print(f"100 argon atoms, canonical: C_V / (N k_B) = {argon.value / (100 * K_B):.3f} "
      f"+- {argon.error / (100 * K_B):.3f}   (equipartition: 1.5)")
four = {drift: transport.energy_fluctuation_cv(
    transport.gas_energy_samples(4, T, M_AR, 20_000, rng, remove_drift=drift), T)
    for drift in (False, True)}
print(f"4 argon atoms: C_V / (N k_B) = {four[False].value / (4 * K_B):.3f} +- "
      f"{four[False].error / (4 * K_B):.3f}; with each copy's net momentum removed "
      f"{four[True].value / (4 * K_B):.3f} +- {four[True].error / (4 * K_B):.3f}   "
      f"((3/2)(N - 1)/N = 1.125)")
fixed = transport.energy_fluctuation_cv(
    transport.gas_energy_samples(100, T, M_AR, 500, rng, fix_temperature=True), T)
print(f"100 argon atoms, microcanonical: C_V / (N k_B) = {fixed.value / (100 * K_B):.1e}")

print(f"\n1000 two-level systems, gap / k_B = {GAP / K_B:.1f} K; heat capacity / (N k_B)")
print("  k_B T / gap    from the noise       by heating (+-5%)    exact (module 12)")
two_level = {}
for ratio in (0.25, 0.42, 1.0, 3.0):
    t = ratio * GAP / K_B
    noise = transport.energy_fluctuation_cv(
        transport.two_level_energy_samples(1000, GAP, t, 5000, rng), t)
    hot = transport.two_level_energy_samples(1000, GAP, 1.05 * t, 5000, rng)
    cold = transport.two_level_energy_samples(1000, GAP, 0.95 * t, 5000, rng)
    heated = (hot.mean() - cold.mean()) / (0.1 * t)
    heated_error = np.sqrt((hot.var(ddof=1) + cold.var(ddof=1)) / 5000) / (0.1 * t)
    exact = 1000 * partition.two_level_heat_capacity(GAP, t)
    two_level[ratio] = (noise, heated, heated_error, exact)
    scale = 1000 * K_B
    print(f"  {ratio:6.2f}     {noise.value / scale:.4f} +- {noise.error / scale:.4f}    "
          f"{heated / scale:.4f} +- {heated_error / scale:.4f}     {exact / scale:.4f}")

הרעש הוא קלורימטר. עבור הגז הוא מראה את ה-$\tfrac32$ לאטום של חלוקת האנרגיה השווה; עבור ארבעה אטומים
שהתנע הכולל שלהם הוסר הוא מראה $\tfrac32(N-1)/N$ לאטום — שלוש דרגות החופש שהוקפאו בהסרת הסחיפה כבר אינן
עוברות פלוקטואציות, והרעש מוציא אותן מהספירה. עבור המערכות הדו-רמתיות הוא עוקב אחר עקומת שוטקי של
מודול 12 בכל טמפרטורה, כמו המדידה הרגילה, אבל מתוך נתונים שנלקחו בטמפרטורה אחת בלבד. כשהגז מוגרל
באופן מיקרו-קנוני אין לאנרגיה שלו שום פלוקטואציה, והמכשיר מראה אפס: הזהות שייכת למערכת *שבמגע עם
אמבט*.

## חלק 3 — ריצה ארוכה יותר

אילו הייתה הפלוקטואציה שגיאה נומרית, דגימות נוספות היו מקטינות אותה. צפו במה שדגימות נוספות עושות
בפועל.

**הניבוי שלכם:** במעבר מ-$100$ ל-$100\,000$ דגימות, מה קורה ל-$\sigma_E$, ומה קורה לאי-הוודאות של
האנרגיה הממוצעת?

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
rng = np.random.default_rng(1802)
t = 0.42 * GAP / K_B
exact_c = 1000 * partition.two_level_heat_capacity(GAP, t)
exact_sigma = np.sqrt(K_B * t**2 * exact_c)
print("  samples    sigma_E / gap    error of <E> / gap    C_V from the noise / exact")
longer = {}
for n in (100, 1000, 10_000, 100_000):
    e = transport.two_level_energy_samples(1000, GAP, t, n, rng)
    c = transport.energy_fluctuation_cv(e, t)
    longer[n] = (e.std(ddof=1), e.std(ddof=1) / np.sqrt(n), c)
    print(f"  {n:7d}      {longer[n][0] / GAP:7.3f}          {longer[n][1] / GAP:7.4f}            "
          f"{c.value / exact_c:.4f} +- {c.error / exact_c:.4f}")
print(f"sqrt(k_B T^2 C_V) / gap = {exact_sigma / GAP:.3f}")

ריצה ארוכה פי אלף השאירה את $\sigma_E$ במקומו — ב-$\sqrt{\kB T^2 C_V}$ — והקטינה רק את אי-הוודאות של
מה שנמדד: את פס השגיאה של האנרגיה הממוצעת, ואת זה של קיבול החום. הפלוקטואציה היא תכונה של המערכת, לא
של הרישום.

## חלק 4 — מדוע מערכות גדולות נראות יציבות

הפלוקטואציה היחסית $\sigma_E/\langle E\rangle$ של $N$ מערכות דו-רמתיות ב-$\kB T = \Delta$, עבור $N$
מ-$16$ עד $4096$.

**הניבוי שלכם:** השיפוע של $\log(\sigma_E/\langle E\rangle)$ כנגד $\log N$.

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
rng = np.random.default_rng(1803)
t = GAP / K_B
sizes = 2 ** np.arange(4, 13)
relative = []
for n in sizes:
    e = transport.two_level_energy_samples(int(n), GAP, t, 4000, rng)
    relative.append(e.std(ddof=1) / e.mean())
relative = np.array(relative)
exponent = scaling_exponent(sizes, relative)
p_excited = partition.two_level_energy(GAP, t) / GAP

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.loglog(sizes, relative, "o", color="#2563eb", label="measured")
ax.loglog(sizes, np.sqrt((1 - p_excited) / (sizes * p_excited)), "k--",
          label="sqrt(k_B T^2 C_V) / U")
ax.set_xlabel("N")
ax.set_ylabel("sigma_E / <E>")
ax.legend()
plt.tight_layout()
plt.show()
print(f"fitted exponent {exponent:.3f}; sigma_E / <E> = {relative[0]:.3f} at N = 16, "
      f"{relative[-1]:.4f} at N = 4096")
print(f"for a mole of them: {np.sqrt((1 - p_excited) / (N_A * p_excited)):.1e}")

המעריך הוא $-\tfrac12$, כפי ששלב 2 גזר מתוך הזהות, ועבור מול אחד האנרגיה חדה עד כדי שני חלקים
ב-$10^{12}$: אי אפשר להבחין בין התיאור הקנוני לבין התיאור המיקרו-קנוני.

## חלק 5 — ענן ההולכים

עשרת אלפים הולכים בלתי תלויים, שכל אחד מהם צועד $\pm\ell$ בכל $\tau$, וכולם משוחררים בראשית.
ההיסטוגרמות שלהם בשלושה זמנים משורטטות כשמעליהן פונקציית גרין של משוואת הדיפוזיה עבור
$D = \ell^2/2\tau$ — מחושבת, לא מותאמת — ולצידן ממוצע ריבוע ההעתק שלהם בצירים לוגריתמיים.

**הניבוי שלכם:** פי כמה רחב יותר הענן ב-$1000\tau$ מאשר ב-$250\tau$?

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
rng = np.random.default_rng(1804)
walkers = transport.walker_trajectories(10_000, 1000, STEP, rng)
D_EXACT = transport.persistent_walk_d(STEP, TAU)
spread = transport.msd(walkers)
lags = np.unique(np.geomspace(10, 1000, 25).astype(int))
cloud_slope = scaling_exponent(lags, spread[lags])

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
x = np.linspace(-120, 120, 600)
for t, colour in ((50, "#94a3b8"), (250, "#2563eb"), (1000, "#d97706")):
    centres, density = sampling.walker_histogram(walkers[:, t], n_bins=120, span=(-120, 120),
                                                 lattice=2 * STEP)
    left.step(centres, density, where="mid", color=colour, label=f"t = {t} tau")
    left.plot(x, transport.diffusion_greens_function(x, t * TAU, D_EXACT), color=colour, ls="--")
left.set_xlabel("x / l")
left.set_ylabel("P(x, t) l")
left.legend()
right.loglog(np.arange(1, 1001), spread[1:], color="k", label="measured")
right.loglog(lags, 2 * D_EXACT * lags * TAU, "--", color="#2563eb", label="2 D t")
right.set_xlabel("t / tau")
right.set_ylabel("<x^2> / l^2")
right.legend()
plt.tight_layout()
plt.show()
widths = {t: walkers[:, t].std() for t in (250, 1000)}
print(f"log-log slope of <x^2> from 10 to 1000 steps: {cloud_slope:.4f}")
print(f"width at 250 and 1000 steps: {widths[250]:.2f} l and {widths[1000]:.2f} l, "
      f"ratio {widths[1000] / widths[250]:.3f}")

פי ארבעה בזמן, פי שניים ברוחב: רוחב הענן גדל כמו $\sqrt{t}$, לממוצע ריבוע ההעתק יש שיפוע אחד,
ופונקציית גרין — ניבוי ללא שום פרמטר חופשי — מונחת על ההיסטוגרמה בכל זמן.

## חלק 6 — $D$ בשלוש דרכים

שלוש מדידות של אותו $D$, המשתמשות בשלושה מאפיינים שונים של ההולכים: הגדילה של ממוצע ריבוע ההעתק
(`d_from_msd`, שנשלט בידי זנבות הענן); התאמה בריבועים פחותים של צורת ההיסטוגרמה לפונקציית גרין
בארבעה זמנים (`d_from_histogram`, שנשלטת בידי האמצע שלה); והאינטגרל של המתאם העצמי של המהירות
(`d_from_vacf`, שאינו משתמש במיקומים כלל). כל פס שגיאה מתקבל מחלוקת ההולכים לעשרים קבוצות בלתי
תלויות.

**הניבוי שלכם:** לאיזו מבין השלוש יהיה פס השגיאה הקטן ביותר, ומדוע?

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
velocities = transport.walker_velocities(walkers, TAU)
three = {
    "mean squared displacement": transport.d_from_msd(walkers, TAU),
    "Green's-function histogram fit": transport.d_from_histogram(walkers, TAU, lattice=2 * STEP),
    "velocity autocorrelation": transport.d_from_vacf(velocities, TAU),
}
for name, estimate in three.items():
    print(f"{name:32s} D = {estimate.value:.4f} +- {estimate.error:.4f}   "
          f"({(estimate.value - D_EXACT) / estimate.error:+.1f} errors from l^2/(2 tau) = 0.5)")
names = list(three)
consistent = {(a, b): three[a].consistent_with(three[b])
              for i, a in enumerate(names) for b in names[i + 1:]}
for (a, b), verdict in consistent.items():
    print(f"  {a} vs {b}: {'consistent' if verdict else 'NOT consistent'} within 2 combined errors")

vacf = transport.velocity_autocorrelation(velocities, 100)
running = transport.green_kubo_running(vacf, TAU)
print("C(k) / C(0) for k = 0 ... 5:", np.round(vacf[:6] / vacf[0], 4))
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(np.arange(101), running, color="k", label="tau [C(0)/2 + C(1) + ... + C(K)]")
ax.axhline(D_EXACT, color="#2563eb", ls="--", label="l^2 / (2 tau)")
ax.set_xlabel("cut-off K (steps)")
ax.set_ylabel("running integral (l^2 / tau)")
ax.set_ylim(0, 0.7)
ax.legend()
plt.tight_layout()
plt.show()

שלושה מאפיינים שונים של הנתונים, $D$ אחד. האומד שעובר דרך המהירות הוא המדויק ביותר כאן, משתי סיבות:
כל ראשית זמן של כל הולך תורמת לו, בעוד שממוצע ריבוע ההעתק משתמש רק בנקודת ההתחלה של כל הולך; וכל צעד
הוא בדיוק $\pm\ell$, ולכן $C(0)$ אינו נושא שום רעש. עבור ההולכים האלה הוא בודק בעיקר שהצעדים בלתי
תלויים — חלק 7 הוא המבחן הקשה יותר. המהירות של הולך בלתי תלוי נשכחת אחרי צעד אחד — $C(k)$ הוא אפס
בתוך הרעש עבור $k \geq 1$ — והאינטגרל הוא $\tfrac12\tau C(0) = \ell^2/2\tau$.

## חלק 7 — הולכים עם זיכרון

עכשיו כל הולך חוזר על הצעד האחרון שלו בהסתברות $q = 0.9$, ובכל מקרה אחר עושה צעד אקראי חדש, כמו
מולקולה השומרת על מהירותה בין התנגשויות: ההילוך המתמיד של מודול 03. הצעדים הם עדיין $\pm\ell$
בהסתברות שווה, אבל כבר אינם בלתי תלויים: המתאם בין צעדים המרוחקים $k$ זה מזה הוא $q^k$.

**הניבוי שלכם:** האם $D = \ell^2/2\tau$ עדיין מתקיים? אם לא, בכמה הוא שוגה?

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
rng = np.random.default_rng(1806)
Q = 0.9
memory = transport.walker_trajectories(5000, 1000, STEP, rng, persistence=Q)
memory_v = transport.walker_velocities(memory, TAU)
D_MEMORY = transport.persistent_walk_d(STEP, TAU, Q)
spread_q = transport.msd(memory)
short_slope = scaling_exponent(np.arange(1, 4), spread_q[1:4])
long_slope = scaling_exponent(np.arange(300, 1001, 50), spread_q[300:1001:50])
three_q = {
    "mean squared displacement": transport.d_from_msd(memory, TAU),
    "Green's-function histogram fit": transport.d_from_histogram(memory, TAU, lattice=2 * STEP),
    "velocity autocorrelation": transport.d_from_vacf(memory_v, TAU),
}
print(f"exact D = (l^2/(2 tau)) (1 + q)/(1 - q) = {D_MEMORY:.2f}; the formula l^2/(2 tau) "
      f"says {D_EXACT}")
for name, estimate in three_q.items():
    print(f"{name:32s} D = {estimate.value:.3f} +- {estimate.error:.3f}   "
          f"({(estimate.value - D_MEMORY) / estimate.error:+.1f} errors from the exact value)")
vacf_q = transport.velocity_autocorrelation(memory_v, 100)
memory_gap = float(np.max(np.abs(vacf_q[:21] / vacf_q[0] - Q ** np.arange(21))))
print(f"log-log slope of <x^2>: {short_slope:.3f} over steps 1-3, {long_slope:.3f} over 300-1000")
print(f"C(k)/C(0) against q^k for k <= 20: largest difference {memory_gap:.4f}")

# One run is one measurement. Repeat the whole experiment under six independent seeds.
repeats = {name: [] for name in three_q}
for seed in np.random.SeedSequence(1816).spawn(6):
    generator = np.random.default_rng(seed)
    trial = transport.walker_trajectories(2000, 1000, STEP, generator, persistence=Q)
    repeats["mean squared displacement"].append(transport.d_from_msd(trial, TAU).value)
    repeats["Green's-function histogram fit"].append(
        transport.d_from_histogram(trial, TAU, lattice=2 * STEP).value)
    repeats["velocity autocorrelation"].append(
        transport.d_from_vacf(transport.walker_velocities(trial, TAU), TAU).value)
print("\nsix independent repeats of 2000 walkers:")
for name, values in repeats.items():
    values = np.array(values)
    print(f"{name:32s} mean {values.mean():.3f} +- {values.std(ddof=1) / np.sqrt(values.size):.3f}")

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
t = np.arange(1, 1001)
left.loglog(t, spread_q[1:], color="k", label="measured")
left.loglog(t, (STEP / TAU) ** 2 * (t * TAU) ** 2, ":", color="#dc2626", label="<v^2> t^2")
left.loglog(t, 2 * D_MEMORY * t * TAU, "--", color="#2563eb", label="2 D t")
left.set_ylim(0.5, 3e4)
left.set_xlabel("t / tau")
left.set_ylabel("<x^2> / l^2")
left.legend()
right.plot(np.arange(41), vacf_q[:41] / vacf_q[0], "o", color="k", ms=4, label="measured")
right.plot(np.arange(41), Q ** np.arange(41), color="#2563eb", label="q^k")
right.set_xlabel("lag k (steps)")
right.set_ylabel("C(k) / C(0)")
right.legend()
plt.tight_layout()
plt.show()

זיכרון המהירות דועך כמו $q^k$, ממוצע ריבוע ההעתק מתחיל בתנועה בליסטית (שיפוע שניים) ועובר למשטר של
דיפוזיה (שיפוע אחד) ברגע שהזיכרון מוצה, ו-$D$ גדול פי תשעה-עשר מ-$\ell^2/2\tau$ — הגורם
$(1+q)/(1-q)$ שמודול 03 מצא בשונות של ההילוך המתמיד. הנוסחה של צעדים בלתי תלויים נכשלת משום שההנחה
שלה נכשלת; האינטגרל של גרין–קובו אינו נכשל, משום שהוא שואל רק כמה זמן המהירות נשמרת בזיכרון. בריצה
הבודדת שלמעלה, שלוש ההערכות אינן חייבות ליפול בתוך פסי השגיאה שלהן זו מזו ומן הערך המדויק — בערך
מדידה אחת מכל עשרים נופלת במרחק של יותר משני פסי שגיאה — ובשביל זה נועדו שש החזרות.

## חלק 8 — הקופסה של מודול 04, שאין בה דיפוזיה

מאתיים אטומי ארגון בקופסה הדו-ממדית של מודול 04, שאורך צלעה $1\ \mu\mathrm{m}$, בלי התנגשויות
ביניהם, נרשמים צעד אחר צעד בעזרת `kinetics.simulate` ללא שינוי.

**הניבוי שלכם:** מה עושה ממוצע ריבוע ההעתק של גז הקופסה בזמנים ארוכים?

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
rng = np.random.default_rng(1807)
SIDE = 1e-6
box_state = kinetics.initialise_gas(200, (SIDE, SIDE), 300.0, M_AR, rng)
box_dt = kinetics.max_stable_dt(box_state)
box_x, box_v = transport.box_gas_trace(box_state, box_dt, 2000)
box_spread = transport.msd(box_x[:, :, 0].T)
box_vacf = transport.velocity_autocorrelation(box_v[:, :, 0].T, 1500)
box_running = transport.green_kubo_running(box_vacf, box_dt)
box_scale = kinetics.mean_speed(300.0, M_AR) * SIDE
saturation = float(box_spread[-400:].mean() / (SIDE**2 / 6))
crossings = 2000 * box_dt * kinetics.mean_speed(300.0, M_AR) / SIDE
print(f"run: {2000 * box_dt * 1e9:.0f} ns, about {crossings:.0f} box crossings")
print(f"late <x^2> / (L^2/6) = {saturation:.3f}")
print(f"C(s)/C(0): lowest {box_vacf.min() / box_vacf[0]:.2f}, "
      f"at s = {box_vacf.argmin() * box_dt * 1e9:.2f} ns")
print(f"running integral of C: largest {box_running.max() / box_scale:.3f} vbar L, "
      f"final {box_running[-1] / box_scale:+.3f} vbar L")

times_ns = np.arange(box_spread.size) * box_dt * 1e9
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
left.plot(times_ns, box_spread / SIDE**2, color="k")
left.axhline(1 / 6, color="#2563eb", ls="--", label="L^2 / 6")
left.set_xlabel("t (ns)")
left.set_ylabel("<x^2> / L^2")
left.legend()
lag_ns = np.arange(box_vacf.size) * box_dt * 1e9
right.plot(lag_ns, box_vacf / box_vacf[0], color="k", label="C(s) / C(0)")
right.plot(lag_ns, box_running / box_scale, color="#d97706", label="running integral / (vbar L)")
right.axhline(0.0, color="grey", lw=1)
right.set_xlabel("s (ns)")
right.legend()
plt.tight_layout()
plt.show()

גז הקופסה עף, פוגע בדופן, ומפסיק להתפשט ב-$L^2/6$ — ממוצע ריבוע המרחק בין שני מיקומים בלתי תלויים
בקופסה. הדפנות הופכות את כיוון המהירות שלו במקום שהיא תישכח: $C(s)$ נעשה שלילי, והאינטגרל שלו חוזר
לאפס. בלי התנגשויות בין החלקיקים אין דיפוזיה, וזהו אופן הכישלון של מודול 04 במבט מבפנים.

## חלק 9 — התנגשויות בגז ממשי

גודל המולקולה בא מה-$b$ של ואן דר ואלס ממודול 02, ש-`gases.vdw_constants_from_critical` מחשבת מתוך
הנקודה הקריטית; הקבועים הקריטיים לקוחים מספר הכימיה המקוון של NIST. אחר כך המהלך החופשי הממוצע וקצב
ההתנגשויות של חנקן ב-$0\,{}^\circ\mathrm{C}$ ובאטמוספרה אחת, ומה הם אומרים להולך שניתנו לו יחידות פיזיקליות: צעד אחד
הוא מעוף חופשי אחד, וזמן צעד אחד הוא הזמן שבין התנגשויות.

**הניבוי שלכם:** כמה קוטרי מולקולה עוברת מולקולת חנקן בין התנגשויות בתנאי החדר — כ-$3$, כ-$30$ או
כ-$300$?

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
# Critical temperature [K], critical pressure [Pa] and molar mass [kg/mol] (NIST Chemistry WebBook).
CRITICAL = {"He": (5.1953, 0.22832e6, 4.002602e-3),
            "N2": (126.192, 3.3958e6, 28.0134e-3),
            "Ar": (150.687, 4.863e6, 39.948e-3)}
DIAMETER = {name: transport.diameter_from_vdw_b(gases.vdw_constants_from_critical(tc, pc)[1])
            for name, (tc, pc, _) in CRITICAL.items()}
for name, d in DIAMETER.items():
    print(f"{name}: van der Waals diameter {d * 1e9:.3f} nm")

n_stp = 101325.0 / (K_B * 273.15)
n2 = transport.kinetic_estimates(n_stp, 273.15, CRITICAL["N2"][2] / N_A,
                                 transport.hard_sphere_cross_section(DIAMETER["N2"]), 2.5 * K_B)
spacing = n_stp ** (-1 / 3)
print(f"\nnitrogen at 0 C and 1 atm: n = {n_stp:.3e} m^-3, mean spacing {spacing * 1e9:.2f} nm")
print(f"mean free path {n2.mean_free_path * 1e9:.0f} nm = {n2.mean_free_path / DIAMETER['N2']:.0f} "
      f"diameters = {n2.mean_free_path / spacing:.0f} spacings")
print(f"mean speed {n2.mean_speed:.0f} m/s, collision rate {n2.collision_rate:.2e} per second")
walker_d = n2.mean_free_path**2 * n2.collision_rate / 2
print(f"\nas a walker with l = lambda and tau = 1/nu: D = l^2/(2 tau) = {walker_d:.2e} m^2/s (1-D)")
print(f"as a 3-D random flight: D = vbar lambda / 3 = {n2.diffusion:.2e} m^2/s")

כ-$270$ קטרים: גז מדולל, כפי שהניח מודול 04, שבו התנגשויות נדירות בקנה המידה של המולקולות ותכופות —
חמישה מיליארד בשנייה — בכל קנה מידה שאדם יכול לראות. הולך חד-ממדי שכל צעד שלו הוא מהלך חופשי ממוצע
שלם מעריך את $D$ ביתר פי $\tfrac32$: בשלושה ממדים רק שליש מהריבוע הממוצע של מעוף נמצא לאורך ציר אחד
כלשהו, וההתפלגות המעריכית של אורכי המעופים מכפילה את הריבוע הממוצע הזה.

## חלק 10 — צמיגות ומוליכות חום, מול המדידה

קובץ הנתונים מכיל את הצמיגות ואת מוליכות החום של הליום, חנקן וארגון ב-$300\ \mathrm{K}$ מ-$10^{-3}$
עד $10\ \mathrm{MPa}$, מתוך נוסחאות הייחוס של NIST — משוואות שהותאמו למדידות רבות, עם אי-ודאות
מוצהרת של $0.5\%$ (צמיגות) ו-$2\%$ (מוליכות חום) עבור חנקן וארגון, ו-$10\%$ ו-$5\%$ עבור הליום. הן
מושוות להערכות הקינטיות של שלב 8, בעזרת הקטרים של ואן דר ואלס מחלק 9, ולמקדמי דיפוזיה עצמית מדודים.
היחס $\kappa/(\eta c_v)$, כאשר $c_v$ לקילוגרם, שווה ל-$1$ בהערכות הגסות, יהא גודל המולקולה אשר יהא.

**הניבוי שלכם:** באיזה שיעור משתנה הצמיגות של חנקן בין $0.1$ ל-$1\ \mathrm{MPa}$ — שינוי של פי עשרה
בצפיפות?

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
nist = read_table("18-gas-transport.csv")
N_300 = 1e5 / (K_B * 300.0)   # number density at 0.1 MPa and 300 K
comparison = {}
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
print("300 K, 0.1 MPa     eta: estimate / NIST    kappa: estimate / NIST    kappa / (eta c_v)    "
      "eta, 0.1 -> 1 MPa")
for gas, colour in (("He", "#d97706"), ("N2", "#2563eb"), ("Ar", "#111827")):
    rows = nist["gas"] == gas
    pressure = nist["pressure_MPa"][rows] * 1e6
    eta, kappa = nist["viscosity_uPa_s"][rows] * 1e-6, nist["thermal_conductivity_W_mK"][rows]
    cv_mass = nist["cv_J_gK"][rows] * 1e3
    i_low, i_high = np.argmin(np.abs(pressure - 1e5)), np.argmin(np.abs(pressure - 1e6))
    c_v = (2.5 if gas == "N2" else 1.5) * K_B
    est = transport.kinetic_estimates(N_300, 300.0, CRITICAL[gas][2] / N_A,
                                      transport.hard_sphere_cross_section(DIAMETER[gas]), c_v)
    comparison[gas] = {
        "eta": est.viscosity / eta[i_low],
        "kappa": est.thermal_conductivity / kappa[i_low],
        "eucken": kappa[i_low] / (eta[i_low] * cv_mass[i_low]),
        "flat": eta[i_high] / eta[i_low] - 1,
    }
    c = comparison[gas]
    print(f"   {gas:3s}               {c['eta']:6.3f}                  {c['kappa']:6.3f}"
          f"                 {c['eucken']:6.3f}            {100 * c['flat']:+.2f}%")
    left.semilogx(pressure, eta * 1e6, color=colour, label=f"{gas} (NIST)")
    left.axhline(est.viscosity * 1e6, color=colour, ls="--", lw=1)
    right.semilogx(pressure, kappa * 1e3, color=colour, label=f"{gas} (NIST)")
    right.axhline(est.thermal_conductivity * 1e3, color=colour, ls="--", lw=1)
for ax in (left, right):
    ax.axvspan(1e5, 1e6, color="#e5e7eb", zorder=0)
    ax.set_xlabel("pressure (Pa)")
    ax.legend()
left.set_ylabel("viscosity (uPa s); dashed: kinetic estimate")
right.set_ylabel("thermal conductivity (mW/(m K))")
right.set_yscale("log")
plt.tight_layout()
plt.show()

# Measured self-diffusion at 273.2 K and one atmosphere, as tabulated by Hirschfelder, Curtiss
# and Bird, Molecular Theory of Gases and Liquids (1954), Table 8.4-13, p. 581 [m^2/s]:
# argon 0.156e-4 +- 0.002e-4 (Winn 1950) and 0.158e-4 +- 0.002e-4 (Hutchinson 1949);
# nitrogen 0.185e-4 +- 0.006e-4 (Winn 1950) and 0.172e-4 +- 0.002e-4 (Winter 1951).
SELF_DIFFUSION = {"Ar": (0.156e-4, 0.158e-4), "N2": (0.185e-4, 0.172e-4)}
for gas, measured in SELF_DIFFUSION.items():
    est = transport.kinetic_estimates(101325.0 / (K_B * 273.2), 273.2, CRITICAL[gas][2] / N_A,
                                      transport.hard_sphere_cross_section(DIAMETER[gas]))
    print(f"self-diffusion of {gas} at 273 K, 1 atm: estimate {est.diffusion:.3e} m^2/s, measured "
          f"{measured[0]:.3e} and {measured[1]:.3e} "
          f"(ratio {est.diffusion / np.mean(measured):.2f})")

לאורך התחום המוצלל, שינוי של פי עשרה בצפיפות, הצמיגות זזה בפחות מאחוז אחד, כפי שניבא מקסוול; היא
עולה רק במגה-פסקלים, שם הגז כבר אינו מדולל. (מתחת ל-$0.1\ \mathrm{MPa}$ העקומות שטוחות מעצם בנייתן:
זו הצורה של גז מדולל שנוסחאות הייחוס מניחות, ולא מבחן שלה.) ההערכות עצמן נכונות עד כדי גורם מסדר
גודל של אחד, וההסכמה הטובה שלהן עבור חנקן וארגון היא בחלקה מזל: המקדם של התאוריה המדוקדקת גדול פי
$1.5$ בערך מה-$\tfrac13$ הגס, והקוטר של ואן דר ואלס קטן במקרה מקוטר ההתנגשות בערך בשיעור שמפצה על
כך; עבור הליום שתי השגיאות מצטברות. היחס $\kappa/(\eta c_v)$ הוא מבחן נקי יותר, משום שהגודל מצטמצם:
ההערכות הגסות אומרות $1$, הגזים החד-אטומיים אומרים $2.51$ — ה-$\tfrac52$ של התאוריה המדוקדקת — משום
שהמולקולות המהירות ביותר נושאות את מרב האנרגיה ועפות הכי רחוק.

## חלק 11 — האטומים של פרן

קובץ הנתונים הוא טבלה מהמאמר של פרן משנת 1909: ההיטלים האופקיים של העתקים בני $30$ שניות של גרגרים
משרף הגמבוג' ברדיוס $0.212\ \mu\mathrm{m}$, שנספרו בידי שודסג בתאים של $1.7\ \mu\mathrm{m}$, עבור
שתי סדרות של $50$ גרגרים. הסדרה הראשונה הייתה במים ב-$17\,{}^\circ\mathrm{C}$, בצמיגות של $0.011$
פואז ($1.1 \times 10^{-3}\ \mathrm{Pa\,s}$); הטמפרטורה של הסדרה השנייה אינה מצוינת. ההיסטוגרמה
מותאמת להתפלגות גאוסית ב-$|x|$ — אותו אומד כמו בחלק 6, מיושם על טבלה — עם פס שגיאה מדגימה חוזרת,
ואחר כך מומרת ל-$D$, ודרך יחס סטוקס–איינשטיין, למספר אבוגדרו.

**הניבוי שלכם:** עד כדי איזה שיעור ימדדו מאתיים העתקים של גרגר שרף את מספר אבוגדרו?

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
perrin = read_table("18-perrin-projections.csv")
edges = np.append(perrin["low_um"], perrin["high_um"][-1]) * 1e-6
rng = np.random.default_rng(1811)
RADIUS, ETA_WATER, T_WATER, INTERVAL = 0.212e-6, 1.1e-3, 290.15, 30.0
xi2 = {series: transport.binned_variance_estimate(edges, perrin[f"{series}_found"], rng)
       for series in ("first", "second")}
for series, estimate in xi2.items():
    print(f"{series} series: {int(perrin[f'{series}_found'].sum())} projections, "
          f"fitted <x^2> = {estimate.value * 1e12:.1f} +- {estimate.error * 1e12:.1f} um^2")
d_perrin = xi2["first"].value / (2 * INTERVAL)
d_perrin_error = xi2["first"].error / (2 * INTERVAL)
n_a = transport.avogadro_from_diffusion(d_perrin, T_WATER, ETA_WATER, RADIUS)
n_a_error = n_a * d_perrin_error / d_perrin
stokes_einstein = transport.einstein_diffusion(T_WATER,
                                               transport.stokes_friction(ETA_WATER, RADIUS))
print(f"first series: D = {d_perrin:.2e} +- {d_perrin_error:.1e} m^2/s; "
      f"Stokes-Einstein with the defined k_B predicts {stokes_einstein:.2e}")
print(f"Avogadro's number: {n_a:.2e} +- {n_a_error:.1e} per mole (defined value {N_A:.6e})")
counts, centres = perrin["first_found"], 0.5 * (edges[:-1] + edges[1:])
readings = {"second moment of the bin centres": np.sum(counts * centres**2) / counts.sum(),
            "(pi/2) <|x|>^2": np.pi / 2 * (np.sum(counts * centres) / counts.sum()) ** 2}
for name, value in readings.items():
    other = transport.avogadro_from_diffusion(value / (2 * INTERVAL), T_WATER, ETA_WATER, RADIUS)
    print(f"  the same table read as its {name}: <x^2> = {value * 1e12:.1f} um^2 "
          f"-> N_A = {other:.2e}")

# Perrin's own <x^2> for the first series at four intervals (p. 77, in square micrometres),
# from which he computed N = 66, 59, 78 and 89 x 10^22.
PERRIN_XI2 = {30: 50.2, 60: 113.5, 90: 128.0, 120: 144.0}
for interval, value in PERRIN_XI2.items():
    n_from_table = transport.avogadro_from_diffusion(value * 1e-12 / (2 * interval), T_WATER,
                                                     ETA_WATER, RADIUS)
    print(f"  {interval:3d} s: <x^2> = {value} um^2 -> N_A = {n_from_table:.2e}")

# Perrin's summary of N from seven emulsions (Atoms, 1916, p. 123): radius [um], N / 10^22.
PERRIN_EMULSIONS = [(0.50, 80), (0.212, 69.5), (0.212, 55), (0.52, 72.5), (5.50, 78),
                    (0.385, 64), (0.367, 68.8)]
values = np.array([n for _, n in PERRIN_EMULSIONS]) * 1e22
print(f"seven emulsions, radii 0.21 to 5.5 um: N = {values.mean():.2e}, spread "
      f"{values.std(ddof=1) / values.mean():.0%}")

# The fitted Gaussian's probability for each bin of |x|: a difference of erf at the edges.
model = np.diff([math.erf(e / math.sqrt(2 * xi2["first"].value)) for e in edges])
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(centres * 1e6, perrin["first_found"], width=1.6, color="#93c5fd",
       label="found (first series)")
ax.plot(centres * 1e6, model / model.sum() * perrin["first_found"].sum(), "o-", color="#2563eb",
        label="fitted Gaussian in |x|")
ax.set_xlabel("|x| (micrometres)")
ax.set_ylabel("displacements")
ax.legend()
plt.tight_layout()
plt.show()

מאתיים עשרים ואחד מספרים שנרשמו בכתב יד ב-1908, ואותו אומד ששימש עבור ההולכים, נותנים מספר אבוגדרו
המתיישב עם הערך המוגדר. ה-$\pm$ הוא רעש ספירה בלבד, ובחירת האומד חשובה לא פחות: המומנט השני של הטבלה
עצמה, וה-$|x|$ הממוצע שלה, נותנים $42$ ו-$45\ \mu\mathrm{m}^2$ במקום ה-$50$ של ההתאמה, משום שההתאמה
נשענת על תא אחד מלא מדי. הערכים של פרן לכל מרווח זמן מפוזרים עוד יותר, וזהו רעש ספירה של ממש בחמישים
גרגרים שאחריהם עקבו במשך שתי דקות. שבעה תחליבים, ברדיוסים מ-$0.2$ עד $5.5\ \mu\mathrm{m}$ ובמסות
השונות זו מזו פי חמישה-עשר אלף, נתנו כולם $N$ בין $5.5$ ל-$8 \times 10^{23}$: מספר שאינו תלוי במה
שמודדים אותו הוא תכונה של העולם.

## חלק 12 — ניסוי דיפוזיה משלכם

את הדיפוזיה בצנצנת מים כמעט אי אפשר לראות: הפרשי טמפרטורה קטנים מניעים הסעה שנושאת צבע סנטימטרים
בתוך דקות. ג'לטין עוצר את ההסעה ומאפשר למולקולות קטנות לעבור דיפוזיה כמעט כמו במים. הכינו חבילת
ג'לטין שקוף בכוס גבוהה והניחו לו להתמצק במשך הלילה; שפכו מעליו שכבה של צבע מאכל בגוון עז; החזיקו את
הכוס בטמפרטורה קבועה וצלמו אותה ליד סרגל כל שעה או שעתיים במשך יממה. בכל זמן, מדדו את העומק שבו הצבע
מגיע לגוון שבחרתם — אותו גוון בכל פעם — והזינו את הזמנים ואת העומקים בתא שלהלן. חזית דיפוזיה מתקדמת
כמו $\sqrt{t}$: השיפוע בגרף לוגריתמי-לוגריתמי של העומק כנגד הזמן אמור להיות $\tfrac12$. הגוון שבחרתם
קובע כפולה לא ידועה של $\sqrt{Dt}$, ולכן את $D$ עצמו מקבלים רק עד כדי הגורם הזה, אבל המעריך אינו
תלוי בו.

In [ ]:
my_hours = []        # for example [1, 2, 4, 8, 16, 24]
my_depths_mm = []    # the depth of your chosen shade at each of those times
if len(my_hours) >= 3:
    hours, depths = np.array(my_hours, dtype=float), np.array(my_depths_mm, dtype=float)
    print(f"log-log slope of depth against time: {scaling_exponent(hours, depths):.2f} "
          f"(diffusion 0.5, steady speed 1)")
    spread = np.sum(hours * 3600 * (depths * 1e-3) ** 2) / np.sum((hours * 3600) ** 2)
    print(f"depth^2 / t = {spread:.2e} m^2/s; for a quarter-strength shade, D = that / 0.910 "
          f"= {spread / 0.910:.1e} m^2/s")
else:
    print("No measurements yet: fill in my_hours and my_depths_mm, then run this cell again.")

### מדידה

צטטו שלוש תוצאות, כל אחת עם אי-הוודאות שלה ועם ההנחות שהיא נשענת עליהן:

- ה-$D$ של ההולכים הבלתי תלויים של חלק 6, בשלוש הדרכים, עם הכרעתכם אם הן עקביות זו עם זו — ומה הייתה
  משמעותה של אי-עקביות.
- קיבול החום של $100$ אטומי ארגון מתוך הרעש של חלק 2, מול חלוקת האנרגיה השווה.
- מספר אבוגדרו מהנתונים של פרן בחלק 11, מול הערך המוגדר
  $6.02214076 \times 10^{23}\ \mathrm{mol^{-1}}$.

## חלק 13 — הבדיקות האוטומטיות

In [ ]:
# 1. The noise is a calorimeter, and only on a bath.
assert abs(argon.value / (100 * K_B) - 1.5) < 4 * argon.error / (100 * K_B)
assert abs(four[True].value / (4 * K_B) - 1.125) < 4 * four[True].error / (4 * K_B)
assert fixed.value < 1e-12 * 100 * K_B
assert all(abs(noise.value - exact) < 4 * noise.error for noise, _, _, exact in two_level.values())

# 2. A longer run measures the fluctuation better; it does not make it smaller.
assert abs(longer[100_000][0] / exact_sigma - 1) < 0.02
assert longer[100_000][1] < longer[100][1] / 25

# 3. sigma_E / <E> ~ N^(-1/2).
assert abs(exponent + 0.5) < 0.05

# 4. Diffusion: slope one, and three independent measurements of l^2/(2 tau).
assert abs(cloud_slope - 1.0) < 0.02
assert all(abs(e.value - D_EXACT) < 4 * e.error for e in three.values())

# 5. Memory: ballistic then diffusive; D inflated by (1 + q)/(1 - q), as Green-Kubo says.
assert short_slope > 1.8 and abs(long_slope - 1.0) < 0.1 and memory_gap < 0.02
assert all(abs(e.value - D_MEMORY) < 4 * e.error for e in three_q.values())
assert all(abs(np.mean(v) - D_MEMORY) / D_MEMORY < 0.05 for v in repeats.values())

# 6. Module 04's box saturates at L^2/6 and never diffuses.
assert abs(saturation - 1) < 0.1 and abs(box_running[-1]) < 0.05 * box_scale

# 7. Nitrogen at room conditions is dilute.
assert 50e-9 < n2.mean_free_path < 150e-9 and n2.mean_free_path / spacing > 10

# 8. Maxwell: a tenfold change of density moves a dilute gas's viscosity by under 1%; the
#    estimates are honest to a factor of order one; kappa / (eta c_v) is 5/2 for monatomic gases.
assert all(abs(c["flat"]) < 0.01 for c in comparison.values())
assert all(0.3 < c["eta"] < 3 and 0.1 < c["kappa"] < 3 for c in comparison.values())
assert abs(comparison["Ar"]["eucken"] - 2.5) < 0.05 and abs(comparison["He"]["eucken"] - 2.5) < 0.05

# 9. Perrin's granules weigh the atom to within their counting noise.
assert abs(xi2["first"].value * 1e12 - PERRIN_XI2[30]) < 2 * xi2["first"].error * 1e12
assert abs(n_a - N_A) < 3 * n_a_error
print("all checks passed")

## חלק 14 — חקרו בעצמכם

ארבעה כלי חקירה; כוונו את הפקדים ולחצו על **Run Interact**.

**ענן ההולכים.** שחררו עד עשרת אלפים הולכים והתבוננו בהיסטוגרמה שלהם בזמן שתבחרו, כשמעליה פונקציית
גרין עבור ה-$D$ של ההולכים בזמנים ארוכים, ובממוצע ריבוע ההעתק שלהם. דברים לנסות: $250$ ואחר כך
$1000$ צעדים; הצירים הלוגריתמיים; אחר כך זיכרון של $0.95$, ותמונת מצב מוקדמת.

In [ ]:
import ipywidgets as widgets


def cloud(n_walkers=4000, snapshot=250, memory=0.0, log_axes=True, seed=1):
    generator = np.random.default_rng(seed)
    x_cloud = transport.walker_trajectories(n_walkers, 1000, STEP, generator, persistence=memory)
    d_long = transport.persistent_walk_d(STEP, TAU, memory)
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
    width = 4 * np.sqrt(2 * d_long * snapshot * TAU) + 10
    centres, density = sampling.walker_histogram(x_cloud[:, snapshot], n_bins=60,
                                                 span=(-width, width), lattice=2 * STEP)
    left.bar(centres, density, width=centres[1] - centres[0], color="#93c5fd")
    grid = np.linspace(-width, width, 400)
    left.plot(grid, transport.diffusion_greens_function(grid, snapshot * TAU, d_long),
              color="#2563eb")
    left.set_xlabel("x / l")
    left.set_ylabel("P(x, t) l")
    t = np.arange(1, 1001)
    measured = transport.msd(x_cloud)[1:]
    if log_axes:
        right.loglog(t, measured, color="k")
        right.loglog(t, 2 * d_long * t * TAU, "--", color="#2563eb")
    else:
        right.plot(t, measured, color="k")
        right.plot(t, 2 * d_long * t * TAU, "--", color="#2563eb")
    right.axvline(snapshot, color="#d97706", lw=1)
    right.set_xlabel("t / tau")
    right.set_ylabel("<x^2> / l^2")
    plt.tight_layout()
    plt.show()
    estimate = transport.d_from_msd(x_cloud, TAU)
    print(f"D from the mean squared displacement: {estimate.value:.3f} +- {estimate.error:.3f}; "
          f"exact {d_long:.3f}")


widgets.interact_manual(
    cloud,
    n_walkers=widgets.IntSlider(value=4000, min=100, max=10_000, step=100),
    snapshot=widgets.IntSlider(value=250, min=5, max=1000, step=5),
    memory=widgets.FloatSlider(value=0.0, min=0.0, max=0.98, step=0.01),
    log_axes=widgets.Checkbox(value=True, description="log axes"),
    seed=widgets.IntText(value=1),
)

**כלי החקירה של הפלוקטואציות.** האנרגיות של $N$ מערכות דו-רמתיות, מוגרלות מההתפלגות הקנונית,
כהיסטוגרמה של $E/N$, עם הפלוקטואציה היחסית מול הניבוי של הזהות. דברים לנסות: $N = 16$ ואחר כך
$N = 4096$; אחר כך טמפרטורה הרבה מתחת לפער.

In [ ]:
def fluctuations(n_systems=64, kt_over_gap=1.0, seed=1):
    generator = np.random.default_rng(seed)
    t = kt_over_gap * GAP / K_B
    energies = transport.two_level_energy_samples(n_systems, GAP, t, 4000, generator)
    c = transport.energy_fluctuation_cv(energies, t)
    exact = n_systems * partition.two_level_heat_capacity(GAP, t)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(energies / (n_systems * GAP), bins=40, color="#93c5fd")
    ax.set_xlabel("E / (N gap)")
    ax.set_ylabel("samples")
    plt.tight_layout()
    plt.show()
    predicted = np.sqrt(K_B * t**2 * exact) / (n_systems * partition.two_level_energy(GAP, t))
    print(f"sigma_E / <E> = {energies.std(ddof=1) / energies.mean():.4f}; "
          f"sqrt(k_B T^2 C_V) / U = {predicted:.4f}")
    print(f"C_V from the noise / (N k_B) = {c.value / (n_systems * K_B):.4f} +- "
          f"{c.error / (n_systems * K_B):.4f}; module 12: {exact / (n_systems * K_B):.4f}")


widgets.interact_manual(
    fluctuations,
    n_systems=widgets.SelectionSlider(options=[int(n) for n in 2 ** np.arange(4, 13)], value=64),
    kt_over_gap=widgets.FloatLogSlider(value=1.0, base=10, min=-1.0, max=1.0, step=0.05),
    seed=widgets.IntText(value=1),
)

**מחשבון ההובלה.** המהלך החופשי הממוצע, קצב ההתנגשויות ושלישיית מקדמי ההובלה של הליום, חנקן או ארגון
בטמפרטורה ובלחץ שתבחרו. דברים לנסות: חנקן מ-$10^5$ ועד $10^3\ \mathrm{Pa}$; ארגון מול הליום באותם
תנאים; ריק של $10^{-4}\ \mathrm{Pa}$, ומספר קנודסן עבור תא בגודל $10\ \mathrm{cm}$.

In [ ]:
def transport_calculator(gas="N2", temperature=300.0, log10_pressure=5.0):
    tc, pc, molar_mass = CRITICAL[gas]
    pressure = 10.0**log10_pressure
    n = pressure / (K_B * temperature)
    c_v = (2.5 if gas == "N2" else 1.5) * K_B
    est = transport.kinetic_estimates(n, temperature, molar_mass / N_A,
                                      transport.hard_sphere_cross_section(DIAMETER[gas]), c_v)
    print(f"{gas} at {temperature:.0f} K and {pressure:.3g} Pa: n = {n:.3e} m^-3")
    print(f"  mean free path {est.mean_free_path:.3e} m; Knudsen number in a 10 cm chamber "
          f"{transport.knudsen_number(est.mean_free_path, 0.1):.2e}")
    print(f"  collision rate {est.collision_rate:.3e} per second; "
          f"mean speed {est.mean_speed:.0f} m/s")
    print(f"  D ~ {est.diffusion:.3e} m^2/s, eta ~ {est.viscosity * 1e6:.2f} uPa s, "
          f"kappa ~ {est.thermal_conductivity * 1e3:.2f} mW/(m K)")


widgets.interact_manual(
    transport_calculator,
    gas=widgets.Dropdown(options=["He", "N2", "Ar"], value="N2"),
    temperature=widgets.FloatSlider(value=300.0, min=50.0, max=1500.0, step=10.0),
    log10_pressure=widgets.FloatSlider(value=5.0, min=-4.0, max=7.0, step=0.25),
)

**מרוץ החזיתות.** שכבת צבע המתפשטת בדיפוזיה אל תוך מים צלולים, מול חזית הנעה במהירות קבועה, ושתיהן
מוצגות בזמן שתבחרו. דברים לנסות: הזמן שבו החזית שבמהירות קבועה עוקפת את חזית הדיפוזיה; אחר כך הכפילו
את הזמן פי ארבעה והשוו את המרחקים.

In [ ]:
def fronts(d_dye=4e-10, steady_speed_mm_per_hour=1.0, hours=4.0):
    t = hours * 3600.0
    height = np.linspace(-5e-3, 25e-3, 600)
    profile = transport.step_profile(height, t, d_dye)
    # The quarter-strength contour of (1/2) erfc(x / 2 sqrt(D t)) sits at 0.954 sqrt(D t).
    diffusive = 0.9539 * np.sqrt(d_dye * t)
    steady = steady_speed_mm_per_hour * 1e-3 * hours
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.fill_between(height * 1e3, profile, color="#93c5fd")
    ax.axvline(diffusive * 1e3, color="#2563eb", ls="--", label="diffusive front (c = c0/4)")
    ax.axvline(steady * 1e3, color="#dc2626", label="steady front")
    ax.set_xlabel("height above the layer (mm)")
    ax.set_ylabel("c / c0")
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f"after {hours:g} h: diffusive front {diffusive * 1e3:.2f} mm, "
          f"steady front {steady * 1e3:.2f} mm")


widgets.interact_manual(
    fronts,
    d_dye=widgets.FloatLogSlider(value=4e-10, base=10, min=-10.0, max=-8.0, step=0.05),
    steady_speed_mm_per_hour=widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1),
    hours=widgets.FloatSlider(value=4.0, min=0.25, max=48.0, step=0.25),
)

## בחנו את הבנתכם

הריצו את התא שלהלן לחידון בבדיקה אוטומטית. אותן שאלות, עם הסברים כתובים לכל אפשרות, נמצאות בעמוד
המודול.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "18-fluctuations-transport.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שתלכו

כתבו כמה משפטים על כל אחת, בתא שלהלן.

1. מה ניבאתם שהתברר כשגוי, ומה בדיוק היה הפגם בהנמקה שלכם?
2. חלק 3 רץ פי אלף יותר זמן, והפלוקטואציה לא קטנה. מה כן קטן, ומדוע זה ההבדל בין תכונה לבין שגיאה?
3. בחלק 7, שלוש ההערכות של $D$ מריצה בודדת אינן חייבות להסכים בתוך פסי השגיאה שלהן. כיצד הייתם
   מכריעים אם אי-הסכמה היא מזל רע או בעיה אמיתית בשיטה?
4. בגז של מודול 04 אין דיפוזיה. מה הייתם צריכים להוסיף למודל שלו כדי שתהיה בו דיפוזיה, ואיזו שורה
   במפרט המודל שלו הייתה משתנה?

**התשובות שלכם:**

1.
2.
3.
4.